In [6]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


OpenML Dataset List of the tabzilla benchmark suite

In [7]:
OPENML_DATASET_IDS = [
    1120,   # Magic telescope
    1053,   # openml__jm1__3904
    4532,   # higgs
    4534,   # phishing websites
    1489,   # phoneme
    1502,   # skin segmentation
    1590,   # adult income
    45072,  # airlines
    151,    # electricity
    4135,   # Amazon_employee_access
    40978,  # internet advertisements
    41434,  # click prediction small
    41150,  # miniBooNE
    40536,  # speeddating
    1043,   # ada agnostic
    1462,   # banknote authentication
    41142,  # christine
    40701,  # churn
    31,     # credit-g
    1471,   # eeg-state
    846,    # elevators
    1038,   # gina agnostic
    821,    # house 16H
    41143,  # jasmine
    1067,   # kc1
    1485,   # madelon
    24,     # mushroom
    1116,   # musk
    1486,   # nomao
    23517,  # numerai28.6
    1487,   # ozone
    1068,   # pc1
    1050,   # pc3
    1049,   # pc4
    41145,  # philippine
    871,    # pollen
    312,    # scene
    38,     # sick
    44,     # spambase
    1570,   # wilt
    45035,  # Albert
    1461,   # bank marketing
    1036,   # sylvia agnostic
    41146,  # sylvine
]

# Step 1 — Minimal-training train/test splits

This block creates the repeated train/test splits for the minimal-training robustness experiment.

Current design:

- 5 repeated stratified random splits.
- No outer validation set.
- The training set contains at least 200 observations from the limiting class.
- The other training class is scaled according to the full-dataset prevalence.
- All remaining observations are assigned to the test set.
- The test set must contain at least 50 positives and 50 negatives.
- The inner train/validation split used for BCE L2 selection is created later.

In [8]:
import numpy as np

MIN_TRAIN_LIMITING_CLASS = 200
MIN_TEST_POS = 50
MIN_TEST_NEG = 50
N_RUNS = 5

SEED_GLOBAL = 1234
MODEL_SEED = 4242

BATCH = 1024
MIN_ROWS = 200

def class_counts(y: np.ndarray) -> tuple[int, int]:
    y = np.asarray(y)
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    return pos, neg


def compute_minimal_train_class_counts(
    prevalence: float,
    min_limiting_class: int = MIN_TRAIN_LIMITING_CLASS,
) -> tuple[int, int]:
    """
    Compute class-specific TRAIN counts.

    The limiting/minority class receives min_limiting_class observations.
    The other class is scaled according to the full-dataset prevalence.

    Examples
    --------
    prevalence = 0.20:
        train_pos = 200
        train_neg = 800

    prevalence = 0.80:
        train_neg = 200
        train_pos = 800
    """
    eps = 1e-12
    p = min(max(float(prevalence), eps), 1.0 - eps)

    if p <= 0.5:
        n_train_pos = int(min_limiting_class)
        n_train_neg = int(np.ceil(min_limiting_class * (1.0 - p) / p))
    else:
        n_train_neg = int(min_limiting_class)
        n_train_pos = int(np.ceil(min_limiting_class * p / (1.0 - p)))

    return n_train_pos, n_train_neg


def make_minimal_train_test_split_for_run(
    y: np.ndarray,
    run_id: int,
    seed_base: int = SEED_GLOBAL,
    min_limiting_class: int = MIN_TRAIN_LIMITING_CLASS,
    min_test_pos: int = MIN_TEST_POS,
    min_test_neg: int = MIN_TEST_NEG,
):
    """
    Create one repeated minimal-training split.

    Strategy
    --------
    1. Compute full-dataset prevalence.
    2. Choose TRAIN class counts:
       - limiting class = min_limiting_class
       - other class scaled by prevalence
    3. Sample TRAIN separately within each class.
    4. Assign all remaining observations to TEST.
    5. Require TEST to contain at least min_test_pos positives
       and min_test_neg negatives.

    Returns
    -------
    train_idx, test_idx, split_info
        If the dataset is infeasible, train_idx and test_idx are None.
    """
    y = np.asarray(y).astype(int)
    n_rows = len(y)

    pos_idx = np.flatnonzero(y == 1)
    neg_idx = np.flatnonzero(y == 0)

    n_pos_total = int(len(pos_idx))
    n_neg_total = int(len(neg_idx))

    if n_rows == 0 or n_pos_total == 0 or n_neg_total == 0:
        split_info = {
            "run_id": int(run_id),
            "seed": None,
            "skip_reason": "not_binary_or_empty",
            "prevalence_full": np.nan,
            "n_rows": int(n_rows),
            "pos_total": int(n_pos_total),
            "neg_total": int(n_neg_total),
            "train_pos_target": None,
            "train_neg_target": None,
            "test_pos_expected": None,
            "test_neg_expected": None,
        }
        return None, None, split_info

    prevalence = float(n_pos_total / n_rows)

    n_train_pos_target, n_train_neg_target = compute_minimal_train_class_counts(
        prevalence=prevalence,
        min_limiting_class=min_limiting_class,
    )

    test_pos_expected = int(n_pos_total - n_train_pos_target)
    test_neg_expected = int(n_neg_total - n_train_neg_target)

    base_info = {
        "run_id": int(run_id),
        "seed": int(seed_base + 1000 * run_id),
        "skip_reason": None,
        "prevalence_full": float(prevalence),
        "n_rows": int(n_rows),
        "pos_total": int(n_pos_total),
        "neg_total": int(n_neg_total),
        "train_pos_target": int(n_train_pos_target),
        "train_neg_target": int(n_train_neg_target),
        "test_pos_expected": int(test_pos_expected),
        "test_neg_expected": int(test_neg_expected),
    }

    if n_train_pos_target > n_pos_total or n_train_neg_target > n_neg_total:
        split_info = {
            **base_info,
            "skip_reason": "not_enough_rows_for_minimal_train",
        }
        return None, None, split_info

    if test_pos_expected < min_test_pos or test_neg_expected < min_test_neg:
        split_info = {
            **base_info,
            "skip_reason": "not_enough_rows_left_for_test",
        }
        return None, None, split_info

    rng = np.random.default_rng(base_info["seed"])

    train_pos_idx = rng.choice(pos_idx, size=n_train_pos_target, replace=False)
    train_neg_idx = rng.choice(neg_idx, size=n_train_neg_target, replace=False)

    train_idx = np.concatenate([train_pos_idx, train_neg_idx])
    rng.shuffle(train_idx)

    train_mask = np.zeros(n_rows, dtype=bool)
    train_mask[train_idx] = True

    test_idx = np.flatnonzero(~train_mask)
    rng.shuffle(test_idx)

    train_pos, train_neg = class_counts(y[train_idx])
    test_pos, test_neg = class_counts(y[test_idx])

    split_info = {
        **base_info,
        "train_pos": int(train_pos),
        "train_neg": int(train_neg),
        "test_pos": int(test_pos),
        "test_neg": int(test_neg),
        "train_n": int(len(train_idx)),
        "test_n": int(len(test_idx)),
    }

    return train_idx, test_idx, split_info

# Step 2 — Shared preprocessing for logistic regression

This block defines the preprocessing pipeline used for logistic regression.

Current design:

- Preprocessing is fit on the outer training set only.
- The test set is transformed using the fitted outer-training preprocessor.
- Categorical variables are grouped using top-K category grouping, followed by one-hot encoding.
- Binary variables are encoded as a single numeric column.
- Continuous numeric variables are median-imputed and standardized.

In [9]:

import numpy as np
import pandas as pd

from dataclasses import dataclass
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import TensorDataset, DataLoader


def _to_dense_float32(X):
    if sparse.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=np.float32)


def _make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse=False)


def infer_feature_types(X: pd.DataFrame, *, numeric_cat_max_unique: int = 20):
    binary_cols, categorical_cols, numeric_cols = [], [], []

    for c in X.columns:
        s = X[c]
        nunq_including_nan = pd.Series(s).nunique(dropna=False)

        if nunq_including_nan == 2:
            binary_cols.append(c)
            continue

        dtype_name = str(s.dtype)

        if dtype_name in ("object", "category", "string"):
            categorical_cols.append(c)
            continue

        if pd.api.types.is_numeric_dtype(s):
            if pd.Series(s).nunique(dropna=False) <= numeric_cat_max_unique:
                categorical_cols.append(c)
            else:
                numeric_cols.append(c)
            continue

        categorical_cols.append(c)

    return binary_cols, categorical_cols, numeric_cols


class TopKCategoryGrouper(BaseEstimator, TransformerMixin):

    def __init__(self, top_k: int = 10):
        self.top_k = int(top_k)
        self.keep_values_ = None
        self.columns_ = None

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)
        self.keep_values_ = {}

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            vc = s.value_counts(dropna=False)
            self.keep_values_[c] = set(vc.head(self.top_k).index.tolist())

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = pd.DataFrame(index=X_df.index)

        for c in self.columns_:
            s = X_df[c].astype("object")
            s = s.where(~pd.isna(s), "__NaN__").astype(str)
            keep = self.keep_values_[c]
            out[c] = s.where(s.isin(keep), "__OTHER__")

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


class BinaryPassthroughEncoder(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.columns_ = None
        self.fill_values_ = {}
        self.value_maps_ = {}

    def _as_frame(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=self.columns_)

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = list(X_df.columns)

        for c in self.columns_:
            s = X_df[c]
            non_missing = s[~pd.isna(s)]

            if len(non_missing) == 0:
                self.fill_values_[c] = 0
                self.value_maps_[c] = {}
                continue

            mode_vals = non_missing.mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else non_missing.iloc[0]
            self.fill_values_[c] = fill_val

            seen = []
            for v in non_missing:
                if v not in seen:
                    seen.append(v)

            if len(seen) == 1:
                mapping = {seen[0]: 0.0}
            else:
                mapping = {seen[0]: 0.0, seen[1]: 1.0}

            self.value_maps_[c] = mapping

        return self

    def transform(self, X):
        X_df = self._as_frame(X)
        out = np.zeros((len(X_df), len(self.columns_)), dtype=np.float32)

        for j, c in enumerate(self.columns_):
            s = X_df[c].copy()
            fill_val = self.fill_values_[c]
            mapping = self.value_maps_[c]

            s = s.where(~pd.isna(s), fill_val)
            default_code = mapping.get(fill_val, 0.0)

            out[:, j] = s.map(lambda v: mapping.get(v, default_code)).astype(np.float32).to_numpy()

        return out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(self.columns_, dtype=object)
        return np.asarray(input_features, dtype=object)


@dataclass
class PreprocessorBundle:
    binary_cols: list
    categorical_cols: list
    numeric_cols: list
    preprocessor_base: ColumnTransformer
    feature_names: list


def _fit_shared_preprocessor_bundle(
    X_tr: pd.DataFrame,
    *,
    numeric_cat_max_unique: int = 20,
    top_k_categories: int = 10,
):

    binary_cols, categorical_cols, numeric_cols = infer_feature_types(
        X_tr, numeric_cat_max_unique=numeric_cat_max_unique
    )

    cat_pipe = Pipeline(
        steps=[
            ("topk", TopKCategoryGrouper(top_k=top_k_categories)),
            ("ohe", _make_ohe()),
        ]
    )

    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    bin_pipe = Pipeline(
        steps=[
            ("binary", BinaryPassthroughEncoder()),
        ]
    )

    preprocessor_base = ColumnTransformer(
        transformers=[
            ("cat", cat_pipe, categorical_cols),
            ("num", num_pipe, numeric_cols),
            ("bin", bin_pipe, binary_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

    preprocessor_base.fit(X_tr)

    try:
        feature_names = list(preprocessor_base.get_feature_names_out())
    except Exception:
        feature_names = [f"x{i}" for i in range(preprocessor_base.transform(X_tr.iloc[:1]).shape[1])]

    return PreprocessorBundle(
        binary_cols=binary_cols,
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        preprocessor_base=preprocessor_base,
        feature_names=feature_names,
    )


def fit_train_preprocessor_and_transform(
    X_tr: pd.DataFrame,
    X_te: pd.DataFrame,
):
    bundle = _fit_shared_preprocessor_bundle(X_tr)

    X_tr_enc = _to_dense_float32(bundle.preprocessor_base.transform(X_tr))
    X_te_enc = _to_dense_float32(bundle.preprocessor_base.transform(X_te))

    if np.isnan(X_tr_enc).any() or np.isnan(X_te_enc).any():
        raise RuntimeError("Preprocessing produced NaNs.")

    return bundle, X_tr_enc, X_te_enc

# Step 3— Global configuration and logistic-regression model

This block defines the global configuration for the logistic-regression small-sample analysis.

Current design:

- Device and random seeds are defined once.
- The experiment uses repeated minimal-training splits rather than outer 5-fold cross-validation.
- Smooth net benefit training uses inverse-temperature annealing with `INVERSE_TEMPS = (1, 4, 10)`.
- Early stopping is based on hard training net benefit.
- Net benefit is evaluated over threshold bands around prevalence and, when relevant, inverse prevalence.
- The logistic-regression model is a single linear layer producing logits.

In [18]:
import random

import numpy as np
import torch
import torch.nn as nn

from nbloss.trainer import set_seed


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

SEED_GLOBAL = 1234
MODEL_SEED = 4242

BATCH = 1024
MIN_ROWS = 200

MIN_TRAIN_LIMITING_CLASS = 200
MIN_TEST_POS = 50
MIN_TEST_NEG = 50
N_RUNS = 5

set_seed(SEED_GLOBAL)

LR_ADAMW = 3e-2
INVERSE_TEMPS = (1.0, 4.0, 10.0)
EPOCHS_PER_TEMP = 300
PATIENCE_HARD = 20

TRAIN_RANGE_POINTS = 11
TEST_RANGE_POINTS = 201

BAND_HALF_WIDTH = 0.025
MID_PREV_LOW, MID_PREV_HIGH = 0.40, 0.60

L2_LAM_GRID = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
INNER_VAL_FRAC = 0.25
MIN_INNER_VAL_POS = 30
MIN_INNER_VAL_NEG = 30


def band_from_t_ref(t_ref: float, half_width: float = BAND_HALF_WIDTH) -> tuple[float, float]:
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)


class TorchLR(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.linear = nn.Linear(d_in, 1, bias=True)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

# Step 4 — Output, loading, batching, and threshold helpers

This block defines utility functions used by the logistic-regression small-sample analysis.

Current design:

- Results are appended safely to CSV files.
- Completed datasets can be tracked in a done-registry.
- OpenML datasets are loaded with their default target.
- Binary targets are encoded as `0/1`.
- PyTorch dataloaders are created from already-preprocessed dense matrices.
- Threshold specifications include prevalence and, when prevalence is outside the mid-prevalence range, inverse prevalence.
- Threshold bands are defined symmetrically around each reference threshold.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import openml
import torch

from torch.utils.data import TensorDataset, DataLoader


# ============================================================================
# Output helpers
# ============================================================================

def _append_csv_safely(df: pd.DataFrame, path: Path):
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def _read_done_registry(path_done: Path) -> set[int]:
    if not path_done.exists():
        return set()

    try:
        df = pd.read_csv(path_done)
        return set(df["openml_id"].astype(int).tolist())
    except Exception:
        return set()


def _mark_dataset_done(openml_id: int, name: str, path_done: Path):
    _append_csv_safely(
        pd.DataFrame([{"openml_id": int(openml_id), "name": str(name)}]),
        path_done,
    )


# ============================================================================
# Dataloader helper
# ============================================================================

def make_loader(X, y, batch=BATCH, shuffle=False, seed=SEED_GLOBAL):
    g = torch.Generator().manual_seed(seed)

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

    return DataLoader(
        ds,
        batch_size=batch,
        shuffle=shuffle,
        generator=g,
    )


# ============================================================================
# OpenML loader
# ============================================================================

def load_openml_dataset(did: int):
    ds = openml.datasets.get_dataset(did)

    X_df, y_raw, _, _ = ds.get_data(
        target=ds.default_target_attribute,
        dataset_format="dataframe",
        include_row_id=False,
        include_ignore_attribute=False,
    )

    y_series = pd.Series(y_raw)

    if (
        y_series.dtype.kind in ("U", "S", "O", "b")
        or str(y_series.dtype).startswith("category")
    ):
        y_cat = y_series.astype("category")
        classes = list(y_cat.cat.categories)

        if len(classes) != 2:
            raise ValueError(
                f"Dataset {did} is not binary after categorical encoding: "
                f"classes={classes}"
            )

        y = y_cat.cat.codes.astype("int64").to_numpy().astype("float32")
    else:
        classes = sorted(pd.Series(y_series).dropna().unique().tolist())

        if len(classes) != 2:
            raise ValueError(
                f"Dataset {did} is not binary: classes={classes}"
            )

        y = y_series.astype("int64").to_numpy().astype("float32")

    prev = float((y == 1).mean())

    meta = dict(
        openml_id=int(ds.dataset_id),
        name=str(ds.name),
        n_rows=int(len(y)),
        n_features=int(X_df.shape[1]),
        prevalence=prev,
        pos_label=(classes[1] if len(classes) == 2 else "1"),
        neg_label=(classes[0] if len(classes) == 2 else "0"),
    )

    return ds, X_df, y, meta


# ============================================================================
# Threshold helpers
# ============================================================================

def threshold_specs_from_prevalence(
    prev: float,
    *,
    mid_prev_low: float = MID_PREV_LOW,
    mid_prev_high: float = MID_PREV_HIGH,
):
    specs = [
        {
            "threshold_name": "prevalence",
            "threshold": float(prev),
        }
    ]

    if prev < mid_prev_low or prev > mid_prev_high:
        specs.append(
            {
                "threshold_name": "inverse",
                "threshold": float(1.0 - prev),
            }
        )

    return specs


def band_from_t_ref(
    t_ref: float,
    *,
    half_width: float = BAND_HALF_WIDTH,
):
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)

# Step 5 — Penalized BCE fitting and L2 selection

This block fits the BCE-trained logistic-regression baseline.

Current design:

- BCE is optimized with LBFGS on a fixed preprocessed design matrix.
- L2 regularization is applied to weights only; the intercept is not penalized.
- The L2 penalty strength is selected using an inner train/validation split within the outer training set.
- After selecting L2, BCE is refit on the full outer training set.

In [16]:
import numpy as np
import torch
import torch.nn as nn


def l2_penalty_weights_only(model: torch.nn.Module) -> torch.Tensor:
    penalty = torch.zeros((), device=next(model.parameters()).device)

    for name, param in model.named_parameters():
        if param.requires_grad and not name.endswith("bias"):
            penalty = penalty + torch.sum(param ** 2)

    return penalty


@torch.no_grad()
def bce_logits_loss(
    model: torch.nn.Module,
    X_t: torch.Tensor,
    y_t: torch.Tensor,
) -> float:
    model.eval()

    bce = torch.nn.BCEWithLogitsLoss(reduction="mean")
    logits = model(X_t).view(-1)
    y_t = y_t.float().view(-1)

    return float(bce(logits, y_t).detach().cpu().item())


def fit_bce_lbfgs(
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    *,
    l2_lambda: float,
    max_iter: int = 500,
    tol_grad: float = 1e-7,
    tol_change: float = 1e-9,
    history_size: int = 100,
    device: str = "cpu",
):
    device_t = torch.device(device)

    X = torch.tensor(X_tr, dtype=torch.float32, device=device_t)
    y = torch.tensor(y_tr, dtype=torch.float32, device=device_t).view(-1)

    model = make_model_fn().to(device_t)
    bce = nn.BCEWithLogitsLoss(reduction="mean")

    opt = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=int(max_iter),
        tolerance_grad=float(tol_grad),
        tolerance_change=float(tol_change),
        history_size=int(history_size),
        line_search_fn="strong_wolfe",
    )

    l2_lambda_t = torch.tensor(float(l2_lambda), device=device_t)

    def closure():
        opt.zero_grad(set_to_none=True)

        logits = model(X).view(-1)
        loss = bce(logits, y) + l2_lambda_t * l2_penalty_weights_only(model)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite LBFGS loss: {loss.detach().item()}")

        loss.backward()
        return loss

    opt.step(closure)

    train_bce = bce_logits_loss(model, X, y)

    return model, train_bce


def select_l2_by_val_bce(
    lambdas,
    *,
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    y_va: np.ndarray,
    device: str = "cpu",
    lbfgs_max_iter: int = 500,
):
    device_t = torch.device(device)

    X_va_t = torch.tensor(X_va, dtype=torch.float32, device=device_t)
    y_va_t = torch.tensor(y_va, dtype=torch.float32, device=device_t).view(-1)

    rows = []
    best_lambda = None
    best_val_bce = float("inf")

    for lam in lambdas:
        model, train_bce = fit_bce_lbfgs(
            make_model_fn,
            X_tr,
            y_tr,
            l2_lambda=float(lam),
            max_iter=int(lbfgs_max_iter),
            device=device,
        )

        val_bce = bce_logits_loss(model, X_va_t, y_va_t)

        rows.append(
            {
                "l2_lambda": float(lam),
                "train_bce": float(train_bce),
                "val_bce": float(val_bce),
            }
        )

        if val_bce < best_val_bce:
            best_lambda = float(lam)
            best_val_bce = float(val_bce)

    return best_lambda, best_val_bce, rows

def make_inner_l2_train_val_split(
    y_train,
    seed: int,
    val_frac: float = INNER_VAL_FRAC,
    min_val_pos: int = MIN_INNER_VAL_POS,
    min_val_neg: int = MIN_INNER_VAL_NEG,
):
    """
    Inner split used only for selecting BCE L2.

    This is not an outer validation set.
    After selecting L2, BCE is refit on the full outer TRAIN set.
    """
    y_train = np.asarray(y_train).astype(int)
    rng = np.random.default_rng(seed)

    pos_idx = np.flatnonzero(y_train == 1)
    neg_idx = np.flatnonzero(y_train == 0)

    n_val_pos = max(min_val_pos, int(round(len(pos_idx) * val_frac)))
    n_val_neg = max(min_val_neg, int(round(len(neg_idx) * val_frac)))

    n_val_pos = min(n_val_pos, len(pos_idx) - 1)
    n_val_neg = min(n_val_neg, len(neg_idx) - 1)

    if n_val_pos < min_val_pos or n_val_neg < min_val_neg:
        raise ValueError(
            "Outer TRAIN set is too small for stable inner L2 validation split: "
            f"n_val_pos={n_val_pos}, n_val_neg={n_val_neg}"
        )

    val_pos_idx = rng.choice(pos_idx, size=n_val_pos, replace=False)
    val_neg_idx = rng.choice(neg_idx, size=n_val_neg, replace=False)

    val_idx = np.concatenate([val_pos_idx, val_neg_idx])
    rng.shuffle(val_idx)

    val_mask = np.zeros(len(y_train), dtype=bool)
    val_mask[val_idx] = True

    inner_train_idx = np.flatnonzero(~val_mask)
    rng.shuffle(inner_train_idx)

    inner_info = {
        "inner_train_pos": int(y_train[inner_train_idx].sum()),
        "inner_train_neg": int(len(inner_train_idx) - y_train[inner_train_idx].sum()),
        "inner_val_pos": int(y_train[val_idx].sum()),
        "inner_val_neg": int(len(val_idx) - y_train[val_idx].sum()),
    }

    return inner_train_idx, val_idx, inner_info

In [13]:
from nbloss.trainer import nb_anneal_only_with_l2
from nbloss.metrics import average_nb_over_range

# Block X — Main runner for minimal-training LR/SNB analysis

This block runs the logistic-regression small-sample robustness experiment.

Current design:

- Each dataset is evaluated using 5 repeated minimal-training splits.
- The training set contains 200 observations from the limiting class, with the other class scaled according to full-dataset prevalence.
- All remaining observations are used as the test set.
- BCE logistic regression is fit with LBFGS.
- L2 regularization is selected using an inner train/validation split within the outer training set.
- BCE is then refit on the full outer training set.
- SNB logistic regression is warm-started from the full-training BCE solution.
- SNB optimization uses smooth net benefit for gradients and hard training net benefit for early stopping.
- Final performance is evaluated only on the test set.

In [19]:
from nbloss.trainer import nb_anneal_only_with_l2
from nbloss.metrics import average_nb_over_range


results = []


def predict_logits_torch(
    model: nn.Module,
    X_np: np.ndarray,
    *,
    device: str = DEVICE,
) -> np.ndarray:
    device_t = torch.device(device)

    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32, device=device_t)

    with torch.no_grad():
        logits = model(X_t).detach().cpu().view(-1).numpy()

    return logits.astype(np.float64)


def evaluate_nb_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    logits_t = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1)
    )
    y_t = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1)
    )

    return float(
        average_nb_over_range(
            logits_t,
            y_t,
            thresh_min=float(thresh_min),
            thresh_max=float(thresh_max),
            num_points=int(num_points),
            input_is_logit=True,
            method="mean",
        )
    )


def add_result_row(
    *,
    openml_id: int,
    dataset_name: str,
    run_id: int,
    threshold_name: str,
    model_name: str,
    t_ref: float,
    t_min: float,
    t_max: float,
    l2_lambda: float,
    inner_val_bce: float,
    train_bce: float,
    test_nb: float,
    delta_vs_bce: float,
    meta: dict,
    split_info: dict,
    inner_info: dict,
    n_features_encoded: int,
):
    row = {
        "model_type": "lr",
        "openml_id": int(openml_id),
        "dataset": str(dataset_name),
        "run_id": int(run_id),

        "threshold_name": str(threshold_name),
        "model": str(model_name),
        "t_ref": float(t_ref),
        "t_min": float(t_min),
        "t_max": float(t_max),

        "test_nb": float(test_nb),
        "delta_vs_bce": float(delta_vs_bce),

        "l2_lambda": float(l2_lambda),
        "inner_val_bce": float(inner_val_bce),
        "train_bce": float(train_bce),

        "n_rows": int(meta["n_rows"]),
        "n_features_raw": int(meta["n_features"]),
        "n_features_encoded": int(n_features_encoded),
        "prevalence_full": float(meta["prevalence"]),
        "pos_label": meta["pos_label"],
        "neg_label": meta["neg_label"],

        "split_seed": int(split_info["seed"]),
        "train_pos_target": int(split_info["train_pos_target"]),
        "train_neg_target": int(split_info["train_neg_target"]),
        "train_pos": int(split_info["train_pos"]),
        "train_neg": int(split_info["train_neg"]),
        "train_n": int(split_info["train_n"]),
        "test_pos": int(split_info["test_pos"]),
        "test_neg": int(split_info["test_neg"]),
        "test_n": int(split_info["test_n"]),
        "pos_total": int(split_info["pos_total"]),
        "neg_total": int(split_info["neg_total"]),

        "inner_train_pos": int(inner_info["inner_train_pos"]),
        "inner_train_neg": int(inner_info["inner_train_neg"]),
        "inner_val_pos": int(inner_info["inner_val_pos"]),
        "inner_val_neg": int(inner_info["inner_val_neg"]),
    }

    results.append(row)
    print(pd.DataFrame([row]))


for openml_id in OPENML_DATASET_IDS:
    print(f"\n==================== OpenML dataset {openml_id} ====================")

    try:
        ds, X_df, y, meta = load_openml_dataset(openml_id)
    except Exception as e:
        print(f"[ERROR] Could not load OpenML dataset {openml_id}: {e}")
        continue

    dataset_name = meta["name"]

    if int(meta["n_rows"]) < int(MIN_ROWS):
        print(
            f"[SKIP] {dataset_name} ({openml_id}) has only "
            f"{meta['n_rows']} rows; MIN_ROWS={MIN_ROWS}."
        )
        continue

    if len(np.unique(y)) != 2:
        print(f"[SKIP] {dataset_name} ({openml_id}) is not binary after loading.")
        continue

    print(
        f"[DATASET] {dataset_name} | "
        f"n={meta['n_rows']} | "
        f"raw_features={meta['n_features']} | "
        f"prevalence={meta['prevalence']:.4f}"
    )

    threshold_specs = threshold_specs_from_prevalence(meta["prevalence"])

    for run_id in range(N_RUNS):
        print(f"\n-------------------- {dataset_name} | run {run_id + 1}/{N_RUNS} --------------------")

        train_idx, test_idx, split_info = make_minimal_train_test_split_for_run(
            y=y,
            run_id=run_id,
            seed_base=SEED_GLOBAL,
            min_limiting_class=MIN_TRAIN_LIMITING_CLASS,
            min_test_pos=MIN_TEST_POS,
            min_test_neg=MIN_TEST_NEG,
        )

        if train_idx is None:
            print(
                f"[SKIP] {dataset_name} ({openml_id}) run={run_id}: "
                f"{split_info.get('skip_reason')} | "
                f"pos_total={split_info.get('pos_total')} | "
                f"neg_total={split_info.get('neg_total')} | "
                f"train_pos_target={split_info.get('train_pos_target')} | "
                f"train_neg_target={split_info.get('train_neg_target')} | "
                f"test_pos_expected={split_info.get('test_pos_expected')} | "
                f"test_neg_expected={split_info.get('test_neg_expected')}"
            )
            continue

        X_tr_df = X_df.iloc[train_idx].copy()
        X_te_df = X_df.iloc[test_idx].copy()

        ytr = np.asarray(y[train_idx], dtype=np.float32).reshape(-1)
        yte = np.asarray(y[test_idx], dtype=np.float32).reshape(-1)

        print(
            f"[SPLIT] seed={split_info['seed']} | "
            f"train(+={split_info['train_pos']}, -={split_info['train_neg']}) | "
            f"test(+={split_info['test_pos']}, -={split_info['test_neg']})"
        )

        inner_tr_rel, inner_va_rel, inner_info = make_inner_l2_train_val_split(
            y_train=ytr,
            seed=int(split_info["seed"]) + 17,
            val_frac=INNER_VAL_FRAC,
            min_val_pos=MIN_INNER_VAL_POS,
            min_val_neg=MIN_INNER_VAL_NEG,
        )

        X_inner_tr_df = X_tr_df.iloc[inner_tr_rel].copy()
        X_inner_va_df = X_tr_df.iloc[inner_va_rel].copy()

        y_inner_tr = ytr[inner_tr_rel]
        y_inner_va = ytr[inner_va_rel]

        _, X_inner_tr, X_inner_va = fit_train_preprocessor_and_transform(
            X_inner_tr_df,
            X_inner_va_df,
        )

        X_inner_tr = np.asarray(X_inner_tr, dtype=np.float32)
        X_inner_va = np.asarray(X_inner_va, dtype=np.float32)

        d_inner = int(X_inner_tr.shape[1])

        def make_inner_lr_model():
            set_seed(MODEL_SEED)
            return TorchLR(d_inner)

        lam_star, best_inner_val_bce, l2_rows = select_l2_by_val_bce(
            L2_LAM_GRID,
            make_model_fn=make_inner_lr_model,
            X_tr=X_inner_tr,
            y_tr=y_inner_tr,
            X_va=X_inner_va,
            y_va=y_inner_va,
            device=DEVICE,
            lbfgs_max_iter=500,
        )

        print(
            f"[L2] selected l2_lambda={lam_star:g} | "
            f"inner VAL BCE={best_inner_val_bce:.6f} | "
            f"inner train(+={inner_info['inner_train_pos']}, -={inner_info['inner_train_neg']}) | "
            f"inner val(+={inner_info['inner_val_pos']}, -={inner_info['inner_val_neg']})"
        )

        bundle, Xtr, Xte = fit_train_preprocessor_and_transform(
            X_tr_df,
            X_te_df,
        )

        Xtr = np.asarray(Xtr, dtype=np.float32)
        Xte = np.asarray(Xte, dtype=np.float32)

        d_in = int(Xtr.shape[1])

        print(
            f"[PREPROC] binary={len(bundle.binary_cols)} | "
            f"categorical={len(bundle.categorical_cols)} | "
            f"numeric={len(bundle.numeric_cols)} | "
            f"encoded_features={d_in}"
        )

        def make_lr_model():
            set_seed(MODEL_SEED)
            return TorchLR(d_in)

        bce_model, train_bce = fit_bce_lbfgs(
            make_lr_model,
            Xtr,
            ytr,
            l2_lambda=float(lam_star),
            max_iter=500,
            device=DEVICE,
        )

        print(f"[BCE] refit on full TRAIN | train BCE={train_bce:.6f}")

        train_dl = make_loader(
            Xtr,
            ytr,
            batch=BATCH,
            shuffle=True,
            seed=int(split_info["seed"]),
        )

        for spec in threshold_specs:
            threshold_name = spec["threshold_name"]
            t_ref = float(spec["threshold"])
            t_min, t_max = band_from_t_ref(t_ref)

            print(
                f"\n[{dataset_name} | run {run_id}] "
                f"=== THRESHOLD: {threshold_name} "
                f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
            )

            logits_bce_test = predict_logits_torch(
                bce_model,
                Xte,
                device=DEVICE,
            )

            nb_bce = evaluate_nb_from_logits_np(
                logits_bce_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(f"[TEST][{threshold_name}] BCE-LR NB={nb_bce:.6f}")

            snb_start = TorchLR(d_in).to(DEVICE)
            snb_start.load_state_dict(
                {
                    k: v.detach().cpu().clone()
                    for k, v in bce_model.state_dict().items()
                }
            )

            snb_model = nb_anneal_only_with_l2(
                snb_start,
                train_dl,
                thresh_min=float(t_min),
                thresh_max=float(t_max),
                num_points_train=int(TRAIN_RANGE_POINTS),
                inverse_temps=tuple(INVERSE_TEMPS),
                epochs_per_temp=int(EPOCHS_PER_TEMP),
                patience_hard=int(PATIENCE_HARD),
                hard_range_num_points=int(TRAIN_RANGE_POINTS),
                lr_adam=float(LR_ADAMW),
                l2_lambda=float(lam_star),
                penalty_fn=l2_penalty_weights_only,
                device=DEVICE,
                seed=int(MODEL_SEED) + 1000 * int(run_id) + 17,
                log_every=20,
            )

            logits_snb_test = predict_logits_torch(
                snb_model,
                Xte,
                device=DEVICE,
            )

            nb_snb = evaluate_nb_from_logits_np(
                logits_snb_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(
                f"[TEST][{threshold_name}] "
                f"SNB-LR NB={nb_snb:.6f} | Δ={nb_snb - nb_bce:+.6f}"
            )

            add_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                model_name="bce_lr",
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                inner_val_bce=best_inner_val_bce,
                train_bce=train_bce,
                test_nb=nb_bce,
                delta_vs_bce=0.0,
                meta=meta,
                split_info=split_info,
                inner_info=inner_info,
                n_features_encoded=d_in,
            )

            add_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                model_name="snb_lr",
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                inner_val_bce=best_inner_val_bce,
                train_bce=train_bce,
                test_nb=nb_snb,
                delta_vs_bce=nb_snb - nb_bce,
                meta=meta,
                split_info=split_info,
                inner_info=inner_info,
                n_features_encoded=d_in,
            )

            if DEVICE == "cuda":
                del snb_model
                torch.cuda.empty_cache()

        if DEVICE == "cuda":
            del bce_model
            torch.cuda.empty_cache()


results_df = pd.DataFrame(results)

print("\nFinished.")
print("results_df shape:", results_df.shape)

try:
    display(results_df.head())
except NameError:
    print(results_df.head())


==================== OpenML dataset 1120 ====================
[DATASET] MagicTelescope | n=19020 | raw_features=10 | prevalence=0.3516

-------------------- MagicTelescope | run 1/5 --------------------
[SPLIT] seed=1234 | train(+=200, -=369) | test(+=6488, -=11963)
[L2] selected l2_lambda=0.01 | inner VAL BCE=0.497937 | inner train(+=150, -=277) | inner val(+=50, -=92)
[PREPROC] binary=0 | categorical=0 | numeric=10 | encoded_features=10
[BCE] refit on full TRAIN | train BCE=0.461650

[MagicTelescope | run 0] === THRESHOLD: prevalence (t_ref=0.3516, [0.3266, 0.3766]) ===
[TEST][prevalence] BCE-LR NB=0.186510
[SNB start] initial train hard NB range = 0.189389
[SNB inverse_temp=1] epoch 020 | train_loss=-0.101722 | train_hard_nb=0.190260
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=1 improved global train hard NB: 0.189389 → 0.194881
[SNB inverse_temp=4] epoch 020 | train_loss=-0.166205 | train_hard_nb=0.198499
[SNB inverse_temp=4] epoch 040 | train_loss=-0.166964 | 

KeyboardInterrupt: 

# Block X — XGBoost BCE baseline helpers

This block defines the XGBoost BCE baseline for the minimal-training robustness experiment.

Current design:

- XGBoost is used only as a BCE/logloss baseline.
- Hyperparameters are selected using an inner train/validation split within the outer training set.
- Early stopping is used only during inner validation hyperparameter selection.
- The final XGBoost model is refit on the full outer training set using the selected hyperparameters and selected number of trees.
- The test set is used only for final evaluation.
- Predictions are returned as logits/margins so net benefit can be evaluated with `input_is_logit=True`.

In [22]:

import xgboost as xgb
import numpy as np

# --- baseline training caps
NUM_BOOST_ROUND_CAP = 4000
EARLY_STOP_ROUNDS   = 100


def _normalize_xgb_params(params: dict) -> dict:
    p = dict(params)

    if "learning_rate" in p:
        p["eta"] = float(p["learning_rate"])

    if "eta" in p and "learning_rate" not in p:
        p["learning_rate"] = float(p["eta"])

    return p


def baseline_hyperparameter_grid():
    """
    Baseline XGBoost hyperparameters are selected by INNER validation logloss only.
    """
    grid = []

    for max_depth in [3, 4]:
        for learning_rate in [0.05, 0.10]:
            for min_child_weight in [1.0, 5.0]:
                for reg_lambda in [1.0, 5.0]:
                    for gamma in [0.0, 1.0]:
                        for subsample in [1.0]:
                            grid.append({
                                "max_depth": int(max_depth),
                                "learning_rate": float(learning_rate),
                                "min_child_weight": float(min_child_weight),
                                "subsample": float(subsample),
                                "colsample_bytree": 1.0,
                                "reg_lambda": float(reg_lambda),
                                "reg_alpha": 0.0,
                                "gamma": float(gamma),
                                "tree_method": "hist",
                            })

    return grid


def xgb_fit_baseline_with_inner_es(
    X_inner_train_enc,
    y_inner_train,
    X_inner_valid_enc,
    y_inner_valid,
    *,
    params: dict,
    seed: int,
    num_boost_round_cap: int = NUM_BOOST_ROUND_CAP,
    early_stopping_rounds: int = EARLY_STOP_ROUNDS,
):
    """
    Train baseline XGBoost with early stopping on INNER validation logloss.

    Returns:
        booster, best_trees, best_logloss
    """

    p = _normalize_xgb_params(params)
    p = dict(p)

    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrain = xgb.DMatrix(X_inner_train_enc, label=y_inner_train)
    dvalid = xgb.DMatrix(X_inner_valid_enc, label=y_inner_valid)

    booster = xgb.train(
        params=p,
        dtrain=dtrain,
        num_boost_round=int(num_boost_round_cap),
        evals=[(dvalid, "inner_valid")],
        verbose_eval=False,
        early_stopping_rounds=int(early_stopping_rounds),
    )

    best_iter = getattr(booster, "best_iteration", None)
    best_score = getattr(booster, "best_score", None)

    if best_iter is None:
        best_iter = int(num_boost_round_cap) - 1

    if best_score is None:
        preds = np.clip(booster.predict(dvalid), 1e-12, 1.0 - 1e-12)
        best_score = float(
            -np.mean(
                y_inner_valid * np.log(preds)
                + (1.0 - y_inner_valid) * np.log(1.0 - preds)
            )
        )

    best_trees = int(best_iter) + 1
    best_logloss = float(best_score)

    return booster, best_trees, best_logloss


def select_xgb_baseline_by_inner_logloss(
    X_inner_train_enc,
    y_inner_train,
    X_inner_valid_enc,
    y_inner_valid,
    *,
    seed: int,
):
    """
    Select XGBoost baseline hyperparameters using INNER validation logloss.

    Returns:
        best_params, best_trees, best_inner_val_logloss, rows
    """

    grid = baseline_hyperparameter_grid()
    best = None

    print(
        f"\n[Inner selection][BCE XGB] "
        f"Evaluating {len(grid)} configs using INNER validation logloss ..."
    )

    rows = []

    for i, cfg in enumerate(grid, start=1):
        booster, best_trees, val_logloss = xgb_fit_baseline_with_inner_es(
            X_inner_train_enc,
            y_inner_train,
            X_inner_valid_enc,
            y_inner_valid,
            params=cfg,
            seed=seed,
        )

        rows.append({
            "grid_i": int(i),
            "inner_val_logloss": float(val_logloss),
            "best_trees": int(best_trees),
            "params": dict(cfg),
        })

        print(
            f"[BCE XGB {i:03d}/{len(grid)}] "
            f"inner_val_logloss={val_logloss:.6f} | "
            f"best_trees={best_trees:4d} | "
            f"max_depth={cfg['max_depth']} | "
            f"lr={cfg['learning_rate']:.2g} | "
            f"min_child_weight={cfg['min_child_weight']} | "
            f"reg_lambda={cfg['reg_lambda']} | "
            f"gamma={cfg['gamma']} | "
            f"subsample={cfg['subsample']} | "
            f"colsample={cfg['colsample_bytree']}"
        )

        if (best is None) or (val_logloss < best[0]):
            best = (float(val_logloss), dict(cfg), int(best_trees))

    best_inner_val_logloss, best_params, best_trees = best

    print(
        f"\n[BCE XGB BEST] inner_val_logloss={best_inner_val_logloss:.6f} | "
        f"best_trees={best_trees} | params={best_params}"
    )

    return best_params, best_trees, best_inner_val_logloss, rows


def refit_xgb_baseline_on_full_train(
    X_train_enc,
    y_train,
    *,
    params: dict,
    n_trees: int,
    seed: int,
):
    """
    Refit final BCE XGBoost on the full outer TRAIN set with fixed #trees.
    """

    p = _normalize_xgb_params(params)
    p = dict(p)

    p.setdefault("objective", "binary:logistic")
    p.setdefault("eval_metric", "logloss")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", int(seed))
    p.setdefault("random_state", int(seed))

    dtrain = xgb.DMatrix(X_train_enc, label=y_train)

    booster = xgb.train(
        params=p,
        dtrain=dtrain,
        num_boost_round=int(n_trees),
        verbose_eval=False,
    )

    return booster


def predict_xgb_logits(booster, X_enc):
    """
    Return logits/margins for NB evaluation with input_is_logit=True.
    """
    dmat = xgb.DMatrix(X_enc)
    logits = booster.predict(dmat, output_margin=True)
    return np.asarray(logits, dtype=np.float32)

# Block X — Main runner for XGBoost BCE baseline

This block runs the XGBoost BCE baseline for the minimal-training robustness experiment.

Current design:

- Each dataset is evaluated using the same 5 repeated minimal-training splits as logistic regression.
- The training set contains 200 observations from the limiting class, with the other class scaled according to full-dataset prevalence.
- All remaining observations are used as the test set.
- XGBoost hyperparameters are selected using an inner train/validation split within the outer training set.
- Early stopping is used only during inner validation hyperparameter selection.
- The final XGBoost model is refit on the full outer training set using the selected hyperparameters and selected number of trees.
- Final performance is evaluated only on the test set.
- XGBoost is included as a BCE/logloss flexibility baseline; no SNB training is applied to XGBoost.

In [23]:

import numpy as np
import pandas as pd
import torch
import xgboost as xgb

def run_dataset_5x_same_preproc(did: int):
    if SKIP_ALREADY_DONE and not FORCE_RERUN:
        done = _read_done_registry()
        if did in done:
            print(f"[SKIP] OpenML {did}: already marked done.")
            return None

    ds, X_df, y, meta = load_openml_dataset(did)

    if meta["n_rows"] < MIN_ROWS:
        print(f"[SKIP {did}] n_rows={meta['n_rows']} < {MIN_ROWS}")
        return None

    _append_csv_safely(pd.DataFrame([meta]), PATH_META)

    prevalence = meta["prevalence"]

    run_specs = []
    t_ref_prev = prevalence
    tmin_prev, tmax_prev = _band_from_t_ref(t_ref_prev, BAND_HALF_WIDTH)
    run_specs.append(("prevalence", t_ref_prev, tmin_prev, tmax_prev))

    if not (MID_PREV_LOW <= prevalence <= MID_PREV_HIGH):
        t_ref_inv = 1.0 - prevalence
        tmin_inv, tmax_inv = _band_from_t_ref(t_ref_inv, BAND_HALF_WIDTH)
        run_specs.append(("inverse", t_ref_inv, tmin_inv, tmax_inv))

    rows = []

    for run_id in range(N_RUNS):
        print(f"\n================ {meta['name']} | XGB run {run_id+1}/{N_RUNS} ================")

        tr_idx, te_idx, split_info = make_minimal_train_test_split_for_run(
            y=y,
            run_id=run_id,
            seed_base=SEED_GLOBAL,
            max_attempts=200,
        )

        if tr_idx is None:
            print(
                f"[SKIP {did}] run={run_id}: {split_info.get('skip_reason')} | "
                f"pos_total={split_info.get('pos_total')} neg_total={split_info.get('neg_total')} | "
                f"train_pos_target={split_info.get('train_pos_target')} "
                f"train_neg_target={split_info.get('train_neg_target')} | "
                f"test_pos_expected={split_info.get('test_pos_expected')} "
                f"test_neg_expected={split_info.get('test_neg_expected')}"
            )
            continue

        X_tr, y_tr = X_df.iloc[tr_idx], y[tr_idx]
        X_te, y_te = X_df.iloc[te_idx], y[te_idx]

        print(
            f"[SPLIT] run={run_id} attempt={split_info['attempt']} seed={split_info['seed']} | "
            f"train(+={split_info['train_pos']},-={split_info['train_neg']}) | "
            f"test(+={split_info['test_pos']},-={split_info['test_neg']})"
        )

        # ---------------------------------------------------------
        # Inner split for XGB hyperparameter selection only
        # ---------------------------------------------------------
        inner_tr_rel, inner_va_rel, inner_info = make_inner_l2_train_val_split(
            y_train=y_tr,
            seed=int(split_info["seed"] + 17),
        )

        X_inner_tr = X_tr.iloc[inner_tr_rel]
        y_inner_tr = y_tr[inner_tr_rel]
        X_inner_va = X_tr.iloc[inner_va_rel]
        y_inner_va = y_tr[inner_va_rel]

        _, X_inner_tr_enc, X_inner_va_enc = fit_inner_preprocessor_and_transform(
            X_inner_tr,
            X_inner_va,
        )

        best_params, best_trees, best_inner_val_logloss, _rows = select_xgb_baseline_by_inner_logloss(
            X_inner_tr_enc,
            y_inner_tr,
            X_inner_va_enc,
            y_inner_va,
            seed=int(MODEL_SEED + run_id),
        )

        print(
            f"[BCE-XGB][run {run_id}] selected by INNER val logloss={best_inner_val_logloss:.6f} | "
            f"best_trees={best_trees} | "
            f"inner train(+={inner_info['inner_train_pos']},-={inner_info['inner_train_neg']}) | "
            f"inner val(+={inner_info['inner_val_pos']},-={inner_info['inner_val_neg']})"
        )

        # ---------------------------------------------------------
        # Final preprocessing fit on full outer TRAIN only
        # ---------------------------------------------------------
        bundle, medians, scaler, X_tr_enc, X_te_enc = fit_train_preprocessor_and_transform(
            X_tr,
            X_te,
        )

        if run_id == 0:
            print(
                f"[PREPROC] binary={len(bundle.binary_cols)} | "
                f"categorical={len(bundle.categorical_cols)} | "
                f"numeric={len(bundle.numeric_cols)} | "
                f"base_dim={X_tr_enc.shape[1]}"
            )

        # ---------------------------------------------------------
        # Final BCE XGB refit on full outer TRAIN
        # ---------------------------------------------------------
        booster_train = refit_xgb_baseline_on_full_train(
            X_tr_enc,
            y_tr,
            params=best_params,
            n_trees=int(best_trees),
            seed=int(MODEL_SEED + run_id),
        )

        test_logits_np = predict_xgb_logits(booster_train, X_te_enc)
        test_logits = torch.tensor(test_logits_np, dtype=torch.float32)

        for run_type, t_ref, t_min, t_max in run_specs:
            nb_bce_band = float(
                average_nb_over_range(
                    test_logits,
                    torch.tensor(y_te, dtype=torch.float32),
                    thresh_min=float(t_min),
                    thresh_max=float(t_max),
                    num_points=int(TEST_RANGE_POINTS),
                    input_is_logit=True,
                    method="mean",
                )
            )

            print(f"[TEST][{run_type}] BCE-XGB TRAIN-only NB over band = {nb_bce_band:.6f}")

            row = dict(
                model_type="xgb",
                openml_id=meta["openml_id"],
                name=meta["name"],
                run_id=run_id,

                split_attempt=int(split_info["attempt"]),
                split_seed=int(split_info["seed"]),
                train_pos=int(split_info["train_pos"]),
                train_neg=int(split_info["train_neg"]),
                test_pos=int(split_info["test_pos"]),
                test_neg=int(split_info["test_neg"]),

                train_pos_target=int(split_info["train_pos_target"]),
                train_neg_target=int(split_info["train_neg_target"]),
                pos_total=int(split_info["pos_total"]),
                neg_total=int(split_info["neg_total"]),

                inner_train_pos=int(inner_info["inner_train_pos"]),
                inner_train_neg=int(inner_info["inner_train_neg"]),
                inner_val_pos=int(inner_info["inner_val_pos"]),
                inner_val_neg=int(inner_info["inner_val_neg"]),

                run_type=run_type,
                threshold=float(t_ref),
                band_min=float(t_min),
                band_max=float(t_max),
                n_rows=meta["n_rows"],
                prevalence=meta["prevalence"],

                nb_nb_band=np.nan,
                nb_bce_band=float(nb_bce_band),
                delta_nb_vs_bce=np.nan,

                best_trees=int(best_trees),
                inner_val_logloss=float(best_inner_val_logloss),
                best_params=str(best_params),

                n_binary_cols=int(len(bundle.binary_cols)),
                n_categorical_cols=int(len(bundle.categorical_cols)),
                n_numeric_cols=int(len(bundle.numeric_cols)),
                n_features_after_preproc=int(X_tr_enc.shape[1]),
            )

            rows.append(row)

    if rows:
        RESULTS_COLUMNS = [
            "model_type",
            "openml_id", "name", "run_id",
            "split_attempt", "split_seed",
            "train_pos", "train_neg", "test_pos", "test_neg",
            "train_pos_target", "train_neg_target",
            "pos_total", "neg_total",
            "inner_train_pos", "inner_train_neg",
            "inner_val_pos", "inner_val_neg",
            "run_type", "threshold", "band_min", "band_max",
            "n_rows", "prevalence",
            "nb_nb_band", "nb_bce_band", "delta_nb_vs_bce",
            "best_trees", "inner_val_logloss", "best_params",
            "n_binary_cols", "n_categorical_cols", "n_numeric_cols",
            "n_features_after_preproc",
        ]

        df_results = pd.DataFrame(rows).reindex(columns=RESULTS_COLUMNS)
        _append_csv_safely(df_results, PATH_RESULTS)

        df_dataset_runs = (
            df_results
            .drop_duplicates(subset=["openml_id", "run_id"])
            .copy()
        )

        df_dataset_runs["train_n"] = df_dataset_runs["train_pos"] + df_dataset_runs["train_neg"]
        df_dataset_runs["test_n"] = df_dataset_runs["test_pos"] + df_dataset_runs["test_neg"]

        size_feature_summary = {
            "openml_id": meta["openml_id"],
            "name": meta["name"],
            "n_runs_completed": int(df_dataset_runs["run_id"].nunique()),
            "mean_n_rows_full_dataset": float(df_dataset_runs["n_rows"].mean()),
            "mean_train_n": float(df_dataset_runs["train_n"].mean()),
            "mean_test_n": float(df_dataset_runs["test_n"].mean()),
            "mean_train_pos": float(df_dataset_runs["train_pos"].mean()),
            "mean_train_neg": float(df_dataset_runs["train_neg"].mean()),
            "mean_test_pos": float(df_dataset_runs["test_pos"].mean()),
            "mean_test_neg": float(df_dataset_runs["test_neg"].mean()),
            "mean_features_after_preproc": float(df_dataset_runs["n_features_after_preproc"].mean()),
        }

        print("\n[DATASET SIZE / FEATURE SUMMARY]")
        for k, v in size_feature_summary.items():
            print(f"{k}: {v}")

        print(f"[SAVED] {meta['name']} → {PATH_RESULTS.name}")
        _mark_dataset_done(meta["openml_id"], meta["name"])

        return df_results

    return None

In [24]:
xgb_results = []


def add_xgb_result_row(
    *,
    openml_id: int,
    dataset_name: str,
    run_id: int,
    threshold_name: str,
    t_ref: float,
    t_min: float,
    t_max: float,
    test_nb: float,
    meta: dict,
    split_info: dict,
    inner_info: dict,
    n_features_encoded: int,
    best_trees: int,
    inner_val_logloss: float,
    best_params: dict,
):
    row = {
        "model_type": "xgb",
        "openml_id": int(openml_id),
        "dataset": str(dataset_name),
        "run_id": int(run_id),

        "threshold_name": str(threshold_name),
        "model": "bce_xgb",
        "t_ref": float(t_ref),
        "t_min": float(t_min),
        "t_max": float(t_max),

        "test_nb": float(test_nb),
        "delta_vs_bce": np.nan,

        "l2_lambda": np.nan,
        "inner_val_bce": np.nan,
        "train_bce": np.nan,

        "best_trees": int(best_trees),
        "inner_val_logloss": float(inner_val_logloss),
        "best_params": str(best_params),

        "n_rows": int(meta["n_rows"]),
        "n_features_raw": int(meta["n_features"]),
        "n_features_encoded": int(n_features_encoded),
        "prevalence_full": float(meta["prevalence"]),
        "pos_label": meta["pos_label"],
        "neg_label": meta["neg_label"],

        "split_seed": int(split_info["seed"]),
        "train_pos_target": int(split_info["train_pos_target"]),
        "train_neg_target": int(split_info["train_neg_target"]),
        "train_pos": int(split_info["train_pos"]),
        "train_neg": int(split_info["train_neg"]),
        "train_n": int(split_info["train_n"]),
        "test_pos": int(split_info["test_pos"]),
        "test_neg": int(split_info["test_neg"]),
        "test_n": int(split_info["test_n"]),
        "pos_total": int(split_info["pos_total"]),
        "neg_total": int(split_info["neg_total"]),

        "inner_train_pos": int(inner_info["inner_train_pos"]),
        "inner_train_neg": int(inner_info["inner_train_neg"]),
        "inner_val_pos": int(inner_info["inner_val_pos"]),
        "inner_val_neg": int(inner_info["inner_val_neg"]),
    }

    xgb_results.append(row)
    print(pd.DataFrame([row]))


for openml_id in OPENML_DATASET_IDS:
    print(f"\n==================== OpenML dataset {openml_id} | XGBoost ====================")

    try:
        ds, X_df, y, meta = load_openml_dataset(openml_id)
    except Exception as e:
        print(f"[ERROR] Could not load OpenML dataset {openml_id}: {e}")
        continue

    dataset_name = meta["name"]

    if int(meta["n_rows"]) < int(MIN_ROWS):
        print(
            f"[SKIP] {dataset_name} ({openml_id}) has only "
            f"{meta['n_rows']} rows; MIN_ROWS={MIN_ROWS}."
        )
        continue

    if len(np.unique(y)) != 2:
        print(f"[SKIP] {dataset_name} ({openml_id}) is not binary after loading.")
        continue

    print(
        f"[DATASET] {dataset_name} | "
        f"n={meta['n_rows']} | "
        f"raw_features={meta['n_features']} | "
        f"prevalence={meta['prevalence']:.4f}"
    )

    threshold_specs = threshold_specs_from_prevalence(meta["prevalence"])

    for run_id in range(N_RUNS):
        print(f"\n-------------------- {dataset_name} | XGB run {run_id + 1}/{N_RUNS} --------------------")

        train_idx, test_idx, split_info = make_minimal_train_test_split_for_run(
            y=y,
            run_id=run_id,
            seed_base=SEED_GLOBAL,
            min_limiting_class=MIN_TRAIN_LIMITING_CLASS,
            min_test_pos=MIN_TEST_POS,
            min_test_neg=MIN_TEST_NEG,
        )

        if train_idx is None:
            print(
                f"[SKIP] {dataset_name} ({openml_id}) run={run_id}: "
                f"{split_info.get('skip_reason')} | "
                f"pos_total={split_info.get('pos_total')} | "
                f"neg_total={split_info.get('neg_total')} | "
                f"train_pos_target={split_info.get('train_pos_target')} | "
                f"train_neg_target={split_info.get('train_neg_target')} | "
                f"test_pos_expected={split_info.get('test_pos_expected')} | "
                f"test_neg_expected={split_info.get('test_neg_expected')}"
            )
            continue

        X_tr_df = X_df.iloc[train_idx].copy()
        X_te_df = X_df.iloc[test_idx].copy()

        ytr = np.asarray(y[train_idx], dtype=np.float32).reshape(-1)
        yte = np.asarray(y[test_idx], dtype=np.float32).reshape(-1)

        print(
            f"[SPLIT] seed={split_info['seed']} | "
            f"train(+={split_info['train_pos']}, -={split_info['train_neg']}) | "
            f"test(+={split_info['test_pos']}, -={split_info['test_neg']})"
        )

        inner_tr_rel, inner_va_rel, inner_info = make_inner_l2_train_val_split(
            y_train=ytr,
            seed=int(split_info["seed"]) + 17,
            val_frac=INNER_VAL_FRAC,
            min_val_pos=MIN_INNER_VAL_POS,
            min_val_neg=MIN_INNER_VAL_NEG,
        )

        X_inner_tr_df = X_tr_df.iloc[inner_tr_rel].copy()
        X_inner_va_df = X_tr_df.iloc[inner_va_rel].copy()

        y_inner_tr = ytr[inner_tr_rel]
        y_inner_va = ytr[inner_va_rel]

        _, X_inner_tr, X_inner_va = fit_train_preprocessor_and_transform(
            X_inner_tr_df,
            X_inner_va_df,
        )

        X_inner_tr = np.asarray(X_inner_tr, dtype=np.float32)
        X_inner_va = np.asarray(X_inner_va, dtype=np.float32)

        best_params, best_trees, best_inner_val_logloss, xgb_grid_rows = (
            select_xgb_baseline_by_inner_logloss(
                X_inner_tr,
                y_inner_tr,
                X_inner_va,
                y_inner_va,
                seed=int(MODEL_SEED) + 1000 * int(run_id) + 29,
            )
        )

        print(
            f"[XGB] selected by INNER val logloss={best_inner_val_logloss:.6f} | "
            f"best_trees={best_trees} | "
            f"inner train(+={inner_info['inner_train_pos']}, -={inner_info['inner_train_neg']}) | "
            f"inner val(+={inner_info['inner_val_pos']}, -={inner_info['inner_val_neg']})"
        )

        bundle, Xtr, Xte = fit_train_preprocessor_and_transform(
            X_tr_df,
            X_te_df,
        )

        Xtr = np.asarray(Xtr, dtype=np.float32)
        Xte = np.asarray(Xte, dtype=np.float32)

        d_in = int(Xtr.shape[1])

        print(
            f"[PREPROC] binary={len(bundle.binary_cols)} | "
            f"categorical={len(bundle.categorical_cols)} | "
            f"numeric={len(bundle.numeric_cols)} | "
            f"encoded_features={d_in}"
        )

        booster = refit_xgb_baseline_on_full_train(
            Xtr,
            ytr,
            params=best_params,
            n_trees=int(best_trees),
            seed=int(MODEL_SEED) + 1000 * int(run_id) + 29,
        )

        logits_xgb_test = predict_xgb_logits(
            booster,
            Xte,
        )

        for spec in threshold_specs:
            threshold_name = spec["threshold_name"]
            t_ref = float(spec["threshold"])
            t_min, t_max = band_from_t_ref(t_ref)

            nb_xgb = evaluate_nb_from_logits_np(
                logits_xgb_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(f"[TEST][{threshold_name}] BCE-XGB NB={nb_xgb:.6f}")

            add_xgb_result_row(
                openml_id=openml_id,
                dataset_name=dataset_name,
                run_id=run_id,
                threshold_name=threshold_name,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                test_nb=nb_xgb,
                meta=meta,
                split_info=split_info,
                inner_info=inner_info,
                n_features_encoded=d_in,
                best_trees=best_trees,
                inner_val_logloss=best_inner_val_logloss,
                best_params=best_params,
            )


xgb_results_df = pd.DataFrame(xgb_results)

print("\nFinished XGBoost.")
print("xgb_results_df shape:", xgb_results_df.shape)

try:
    display(xgb_results_df.head())
except NameError:
    print(xgb_results_df.head())


==================== OpenML dataset 1120 | XGBoost ====================
[DATASET] MagicTelescope | n=19020 | raw_features=10 | prevalence=0.3516

-------------------- MagicTelescope | XGB run 1/5 --------------------
[SPLIT] seed=1234 | train(+=200, -=369) | test(+=6488, -=11963)

[Inner selection][BCE XGB] Evaluating 32 configs using INNER validation logloss ...
[BCE XGB 001/32] inner_val_logloss=0.428462 | best_trees= 137 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=1.0 | gamma=0.0 | subsample=1.0 | colsample=1.0
[BCE XGB 002/32] inner_val_logloss=0.425750 | best_trees= 126 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=1.0 | gamma=1.0 | subsample=1.0 | colsample=1.0
[BCE XGB 003/32] inner_val_logloss=0.411706 | best_trees= 194 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=5.0 | gamma=0.0 | subsample=1.0 | colsample=1.0
[BCE XGB 004/32] inner_val_logloss=0.416057 | best_trees= 137 | max_depth=3 | lr=0.05 | min_child_weight=1.0 | reg_lambda=5

KeyboardInterrupt: 

# Block X — XGBoost in-memory summary

This block summarizes the XGBoost BCE baseline results already produced by the XGBoost runner.

Current design:

- No files are written.
- No done-registry is used.
- Results are read directly from `xgb_results_df`.
- The summary reports mean test net benefit by dataset and threshold.
- Dataset-size and feature-count summaries are computed from the in-memory result dataframe.

In [ ]:
# ======================================================================
# Block X — XGBoost in-memory summary
# Assumes the XGBoost runner already created xgb_results_df.
# Does not save files.
# ======================================================================

import pandas as pd


if "xgb_results_df" not in globals():
    raise NameError(
        "xgb_results_df does not exist. Run the XGBoost runner block first."
    )

df_xgb = xgb_results_df.copy()

if df_xgb.empty:
    print("[AGG] xgb_results_df is empty.")
else:
    print("\nModel counts:")
    print(df_xgb["model_type"].value_counts(dropna=False))

    df_xgb = df_xgb[df_xgb["model_type"] == "xgb"].copy()

    if df_xgb.empty:
        print("[AGG] No XGBoost rows found.")
    else:
        agg_xgb = (
            df_xgb
            .groupby(["openml_id", "dataset", "threshold_name"], as_index=False)
            .agg(
                n_runs=("run_id", "nunique"),
                prevalence_full=("prevalence_full", "first"),
                mean_test_nb=("test_nb", "mean"),
                sd_test_nb=("test_nb", "std"),
                median_best_trees=("best_trees", "median"),
            )
            .sort_values(["openml_id", "threshold_name"])
        )

        print("\nXGBoost BCE-only summary:")
        print(agg_xgb.round(6).to_string(index=False))

        df_run_level_xgb = (
            df_xgb
            .drop_duplicates(subset=["model_type", "openml_id", "run_id"])
            .copy()
        )

        size_summary_xgb = pd.Series({
            "n_openml_datasets": df_run_level_xgb["openml_id"].nunique(),
            "n_openml_runs": len(df_run_level_xgb),
            "mean_full_dataset_n_rows": df_run_level_xgb["n_rows"].mean(),
            "median_full_dataset_n_rows": df_run_level_xgb["n_rows"].median(),
            "mean_train_n": df_run_level_xgb["train_n"].mean(),
            "median_train_n": df_run_level_xgb["train_n"].median(),
            "mean_test_n": df_run_level_xgb["test_n"].mean(),
            "median_test_n": df_run_level_xgb["test_n"].median(),
            "mean_train_pos": df_run_level_xgb["train_pos"].mean(),
            "mean_train_neg": df_run_level_xgb["train_neg"].mean(),
            "mean_test_pos": df_run_level_xgb["test_pos"].mean(),
            "mean_test_neg": df_run_level_xgb["test_neg"].mean(),
            "mean_features_after_preproc": df_run_level_xgb["n_features_encoded"].mean(),
            "median_features_after_preproc": df_run_level_xgb["n_features_encoded"].median(),
        })

        print("\n[OVERALL XGBOOST DATASET SIZE / FEATURE SUMMARY]")
        print(size_summary_xgb)

        try:
            display(agg_xgb)
            display(size_summary_xgb.to_frame("value"))
        except NameError:
            pass